# Clustering

This is an example of capturing BreatheCam footage over a period of time and processing it in order to produce regions of interest. A region of interest is a *large enough* temporal contour containing pixels that are approximately close in color.

### Extra Packages


* `scikit-image`

In [ ]:
import datetime
import numpy as np
import skimage


from breathecam import BreatheCam
from boundaries import BoundingRect
from clustering import cluster

## Generating a BreatheCam Capture

In this example we will process a video pulled from a BreatheCam near Clairton Coke Works.

In [ ]:
location="Clairton Coke Works"
day = datetime.date.fromisoformat("2024-05-19")
camera = BreatheCam(location, day)
time = datetime.time.fromisoformat("09:50:00")
nframes = 40
subsample_factor = 4
view = BoundingRect(3000, 1500, 6500, 2500)
video_rgb_u8 = camera.extract(time, nframes, view=view, subsample_factor=subsample_factor)

## Preprocessing

As smoke spreads, it dissipates, becoming more transparent and less discernable from the background and other foreground objects in the scene. We preprocess the video, attempting to make this dissipation more apparent.

In [ ]:
kernel_size=(128, 128)

video_gray_f32 = np.array([skimage.color.rgb2gray(f) for f in video_rgb_u8])
video_gray_contrast_adapted_f32 = np.array([skimage.exposure.equalize_adapthist(frame, kernel_size) for frame in video_gray_f32])

## Background Subtraction

The clustering algorithm relies on a mask that indicates which pixels to consider as the algorithm progresses. To generate that mask, we use background subtraction, and cluster only those pixels predicted as foreground.

The following algorithm predicts foreground pixels using color difference in reference to a given background model. Other background subtraction algorithms exist and can be substituted for this one.

The color difference formula used is $$\Delta E_{HyAB} = |L_2 - L_1| + \sqrt{(a_2 - a_1)^2 + (b_2 + b_1)^2}$$ The expected color space of the input is the CIELAB color space.

In [ ]:
def get_foreground(video: np.ndarray, bg_model: np.ndarray, threshold: float) -> np.ndarray:
    foregrounds = np.zeros(video.shape[:3])

    for i, frame in enumerate(video):
        difference = np.abs(bg_model - frame)
        distance = difference[..., 0] + np.sqrt(difference[..., 1] ** 2 + difference[..., 2] ** 2)
        foregrounds[i] = distance > threshold

    return foregrounds

Before generating foreground masks, we convert the preprocessed video to the CIELab color space.

In [ ]:
video_cielab_f32 = np.array([
    skimage.rgb2lab(skimage.color.gray2rgb(f)) for f in video_gray_contrast_adapted_f32
])

In [ ]:
bg_threshold = 10
bg_model = np.average(video_cielab_f32, axis=0)
foreground = get_foreground(video_cielab_f32, bg_model, bg_threshold)

To build a contour, we examine and compare neighboring pixels. Neighbors are determined by an array of distances from a center pixel. Contours are temporal, meaning, neighbors can also look at pixels from past frames. For this example, we look at the 8 pixels surrounding the center and the same pixel across 4 past frames.

In [ ]:
NEIGHBORS = np.array([
    (0, 0, -1), (0, -1, -1), (0, -1,  0), (0, -1,  1),
    (0, 0,  1), (0,  1,  1), (0,  1,  0), (0,  1,  1),
    (-1, 0, 0), (-2, 0, 0),(-3, 0, 0), (-4, 0, 0)
])

The last parameter we set is the threshold for whether two pixels are close enough in color, as determined by $$\Delta E_{HyAB} = |L_2 - L_1| + \sqrt{(a_2 - a_1)^2 + (b_2 + b_1)^2}$$

Since our images are in the CIELAB color space, $L \in [0, 100]$, $a \in [-127, 128]$, and $b \in [-128, 127]$. This means that $\Delta E_{HyAB} \in [0, \sim 460.62]$.

In [ ]:
cluster_threshold = 20

The expected colorspace for the video is the CIELAB colorspace. The foreground should be a video of masks.

In [ ]:
temporal_contours = list(cluster(
    video_cielab_f32,
    foreground,
    NEIGHBORS,
    cluster_threshold
))

In [ ]:
min_height = 8
min_width = 8

regions_of_interest = [
    contour for contour in temporal_contours 
    if contour.height > min_height and contour.width > min_width
]

### Generating Minimum Bounded Regions

In [ ]:
minimum_bounding_box_visual = video_rgb_u8.copy()
frame_shape = minimum_bounding_box_visual[0].shape

for contour in regions_of_interest:
    # Generate random color for bounding box
    color = tuple(int(c) for c in np.random.randint(0, 256, size=(3,)))
    bounding_box = contour.minimum_bounding_box()

    for frame in contour.frames:
        x, y = skimage.draw.rectangle_perimeter(
             start=(bounding_box.top, bounding_box.left), 
             end=(bounding_box.bottom, bounding_box.right), 
             shape=frame_shape
        )
        
        minimum_bounding_box_visual[frame][x, y] = color
        

In [ ]:
minimum_bounding_circle_visual = video_rgb_u8.copy()
frame_shape = minimum_bounding_circle_visual[0].shape

for contour in regions_of_interest:
    # Generate random color for bounding circle
    color = tuple(int(c) for c in np.random.randint(0, 256, size=(3,)))
    bounding_circle = contour.minimum_bounding_circle()

    for frame in contour.frames:
        x, y = skimage.draw.circle_perimeter(
             bounding_circle.y,
             bounding_circle.x, 
             radius=bounding_circle.radius,
             shape=frame_shape
        )
        
        minimum_bounding_circle_visual[frame][x, y] = color